# 资源可用性成本问题 (RACP)

**类别：** 调度

来源： [https://www.hexaly.com/templates/resource-availability-cost-problem-racp](https://www.hexaly.com/templates/resource-availability-cost-problem-racp)


## 问题

**在 Resource Availability Cost Problem (RACP) 中**，一个项目由一组需要调度的任务组成。每个任务都有一个给定的持续时间，且不能被中断。任务之间存在优先级约束：每个任务必须在其所有后继任务开始之前结束。问题涉及一组可再生资源。每个任务对每种资源都有一个给定的资源需求或权重（可能为零），表示该任务在执行过程中消耗的资源量。每种资源都有一个需要设定的最大容量。每单位容量都有一个给定的成本，对每种资源有所不同。被处理任务的权重之和不能超过该最大容量。目标是在确保所有任务在给定截止日期之前完成的前提下，找到一个使所需容量的总成本最小的调度方案。

	

### 学到的建模原则

- 添加 [interval decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模任务
- 添加 [integer decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#integer-decisions) 来建模容量
- 定义 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来建模累积资源约束


## 数据

我们提供的 Resource Availability Cost Problem (RACP) 实例来自 [RACP instances](https://www.projectmanagement.ugent.be/research/project_scheduling/racp)，并遵循 Patterson 格式：

- 第一行：

- 任务数量（包括两个额外的持续时间为 0 的虚拟任务：源和汇）
- 可再生资源的数量
- 第二行：每种资源的单位容量成本
- 从第三行开始，对于每个任务：

- 任务的持续时间
- 每种资源的资源需求（权重）
- 后继任务的数量
- 每个后继任务的 ID


## 模型

Resource Availability Cost Problem (RACP) 的 Hexaly 模型使用 interval decision variables 来表示任务。每个 interval 的长度等于相应任务的持续时间。我们还定义了 integer decision variables 来表示分配给每种资源的容量。然后我们写出优先级约束：每个任务必须在其任何后继任务开始之前结束。完工时间（makespan）是所有任务完成的时间，它必须保持小于截止日期。

累积资源约束可以表述如下：对于每种资源以及每个时间槽 t，正在处理的任务所消耗的资源量不能超过该资源的容量。为了建模这些约束，我们对每种资源和每个时间槽的所有活跃任务的权重进行求和。我们使用可变参数的 **and** 公式结合 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，以确保资源容量在任何时刻都被满足。得益于这种可变参数的 **and**，即使时间范围非常大，约束公式仍然紧凑而高效。

最后，我们将总成本计算为每种资源的单位成本与其所选容量的乘积之和。这就是我们希望最小化的目标。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


# The input files follow the "Patterson" format
def read_instance(filename):
    with open(filename) as f:
        lines = f.readlines()

    first_line = lines[0].split()

    # Number of tasks
    nb_tasks = int(first_line[0])

    # Number of resources
    nb_resources = int(first_line[1])

    # Cost per unit of capacity of each resource
    resource_cost = [int(lines[1].split()[r]) for r in range(nb_resources)]

    # Duration of each task
    duration = [0 for _ in range(nb_tasks)]

    # Weight of resource r required for task i
    weight = [[] for _ in range(nb_tasks)]

    # Number of successors
    nb_successors = [0 for _ in range(nb_tasks)]

    # Successors of each task
    successors = [[] for _ in range(nb_tasks)]

    for i in range(nb_tasks):
        line = lines[i + 2].split()
        duration[i] = int(line[0])
        weight[i] = [int(line[r + 1]) for r in range(nb_resources)]
        nb_successors[i] = int(line[nb_resources + 1])
        successors[i] = [int(line[nb_resources + 2 + s]) - 1 for s in range(nb_successors[i])]

    # Trivial upper bound for the end times of the tasks
    horizon = sum(duration[i] for i in range(nb_tasks))

    return (nb_tasks, nb_resources, resource_cost, duration, weight, nb_successors, successors, horizon)


def main(instance_file, dline, output_file, time_limit):
    nb_tasks, nb_resources, resource_cost, duration, weight, nb_successors, successors, horizon = read_instance(
        instance_file)
    
    if dline is None:
        deadline = horizon
    else:
        deadline = int(dline)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Interval decision variables: time range of each task
        tasks = [model.interval(0, horizon) for _ in range(nb_tasks)]

        # Integer decision variables: capacity of each renewable resource
        capacity = [model.int(0, sum(weight[i][r] for i in range(nb_tasks))) for r in range(nb_resources)]

        # Task duration constraints
        for i in range(nb_tasks):
            model.constraint(model.length(tasks[i]) == duration[i])

        # Precedence constraints between the tasks
        for i in range(nb_tasks):
            for s in range(nb_successors[i]):
                model.constraint(model.end(tasks[i]) <= model.start(tasks[successors[i][s]]))

        # Deadline constraint on the makespan
        makespan = model.max([model.end(tasks[i]) for i in range(nb_tasks)])
        model.constraint(makespan <= deadline)

        # Cumulative resource constraints
        for r in range(nb_resources):
            capacity_respected = model.lambda_function(
                lambda t: model.sum(weight[i][r] * model.contains(tasks[i], t)
                                    for i in range(nb_tasks))
                <= capacity[r])
            model.constraint(model.and_(model.range(makespan), capacity_respected))

        # Minimize the total cost
        total_cost = model.sum(resource_cost[r] * capacity[r] for r in range(nb_resources))
        model.minimize(total_cost)
    
        model.close()
        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - total cost
        # - makespan
        # - the capacity of each resource
        # - for each task, the task id, the start and end times
        #
        if output_file != None:
            with open(output_file, "w") as f:
                print("Solution written in file", output_file)
                f.write(str(total_cost.value) + "\n")
                f.write(str(makespan.value) + "\n")
                for r in range(nb_resources):
                    f.write(str(capacity[r].value) + " ")
                f.write("\n")
                for i in range(nb_tasks):
                    f.write(str(i + 1) + " " + str(tasks[i].value.start()) + " " + str(tasks[i].value.end()))
                    f.write("\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python racp.py instance_file [output_file] [time_limit] [deadline]")
        sys.exit(1)
    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    dline = sys.argv[4] if len(sys.argv) >= 5 else None
    main(instance_file, dline, output_file, time_limit)
